# CAER to NCAER-S

## 1. Imports

In [ ]:
import random
from pathlib import Path

import cv2
import pandas as pd
from tqdm.notebook import tqdm

## 2. Config

In [ ]:
CAER_ROOT = Path("datasets/CAER")
OUTPUT_ROOT = Path("datasets/NCAERS")

SEGMENT_DURATION_S = 2.0
RANDOM_SEED = 42

SPLITS = {
    "train": CAER_ROOT / "train",
    "validation": CAER_ROOT / "validation",
    "test": CAER_ROOT / "test",
}

random.seed(RANDOM_SEED)

## 3. Video inventory

In [ ]:
video_records = []

for split_name, split_dir in SPLITS.items():
    for emotion_dir in sorted(split_dir.iterdir()):
        if not emotion_dir.is_dir():
            continue
        label = emotion_dir.name
        for video_path in sorted(emotion_dir.glob("*.avi")):
            video_records.append({
                "split": split_name,
                "label": label,
                "video_path": video_path,
                "video_stem": video_path.stem,
            })

df_videos = pd.DataFrame(video_records)
print(df_videos.groupby(["split", "label"]).size().unstack(fill_value=0))

## 3. Fotogram extraction

In [ ]:
def extract_frames_from_video(video_path, segment_s, rng):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return []

    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    seg_len = max(1, round(fps * segment_s))

    result = []
    seg_buf = []
    frame_no = 0

    while True:
        ok, frame = cap.read()
        if not ok:
            break
        seg_buf.append((frame_no, frame))
        frame_no += 1
        if len(seg_buf) == seg_len:
            result.append(rng.choice(seg_buf))
            seg_buf = []

    if seg_buf:
        result.append(rng.choice(seg_buf))

    cap.release()
    return result


rng = random.Random(RANDOM_SEED)
records = []
errors  = []

for _, row in tqdm(df_videos.iterrows(), total=len(df_videos), desc="Processed videos"):
    split = row["split"]
    label = row["label"]
    video_path = row["video_path"]
    stem = row["video_stem"]

    out_dir = OUTPUT_ROOT / split / label
    out_dir.mkdir(parents=True, exist_ok=True)

    extracted = extract_frames_from_video(video_path, SEGMENT_DURATION_S, rng)

    if not extracted:
        errors.append({"split": split, "label": label, "video": str(video_path)})
        continue

    for seg_idx, (frame_no, frame) in enumerate(extracted):
        out_name = f"{stem}_seg{seg_idx:03d}_f{frame_no:05d}.jpg"
        out_path = out_dir / out_name
        cv2.imwrite(str(out_path), frame)
        records.append({
            "split": split,
            "label": label,
            "video_stem": stem,
            "segment": seg_idx,
            "frame_no": frame_no,
            "filename": out_name,
            "path": out_path.as_posix(),
        })

df_out = pd.DataFrame(records)
print(f"\nFotograms extracted: {len(df_out)}")
if errors:
    print(f"Errors ({len(errors)} videos):", errors[:5])

## 4. Emotion distribution

In [ ]:
summary = df_out.groupby(["split", "label"]).size().unstack(fill_value=0)
summary["TOTAL"] = summary.sum(axis=1)
print(summary)
print(f"\nTotal imgs: {len(df_out)}")

## 5. Save index

In [ ]:
index_path = OUTPUT_ROOT / "ncaers_index.parquet"
df_out.to_parquet(index_path, index=False)
print(f"Index saved in: {index_path}")

if errors:
    err_path = OUTPUT_ROOT / "ncaers_errors.csv"
    pd.DataFrame(errors).to_csv(err_path, index=False)
    print(f"Errors saved in: {err_path}")